# 💳 Credit Card Fraud Detection — Cleaned Up

Rebuilt version. What changed vs. the original and why:

- **Split first, transform second.** Train/test split happens before any scaling or resampling, so nothing from the test set leaks into training.
- **No manual re-scaling of `Amount`/`Time`.** Every model is wrapped in an `sklearn.Pipeline`, so the exact same fitted scaler is applied at train time and predict time. This fixes the main bug from the original notebook: models trained on unscaled `Amount`/`Time` were being evaluated against scaled `Amount`/`Time`, which alone explained most of the terrible precision.
- **Imbalance handled without leaking the test set.** `class_weight='balanced'` / `scale_pos_weight` are used instead of undersampling the whole dataset before splitting. SMOTE (when used) is done inside the pipeline, fit only on the training fold.
- **One clean evaluation set.** Every model is scored on the same untouched, realistically-imbalanced test set — not a balanced one — so the numbers reflect real-world performance.
- **Threshold chosen deliberately**, using the precision-recall curve, instead of defaulting to 0.5.
- **Final artifact is a single pipeline** (scaler + model bundled together), so it's impossible to accidentally apply the wrong scaler at inference time.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_recall_curve,
    average_precision_score, PrecisionRecallDisplay
)

RANDOM_STATE = 42

## Load data

In [ ]:
df = pd.read_csv('creditcard.csv')
df.head()
# Time: seconds elapsed since first transaction
# V1-V28: anonymized PCA features
# Amount: transaction amount (EUR)
# Class: 0 = legit, 1 = fraud

In [ ]:
print(df['Class'].value_counts())
print(f"Fraud rate: {df['Class'].mean():.4%}")

sns.countplot(x='Class', data=df)
plt.title('Class Distribution: Legit (0) vs Fraud (1)')
plt.show()

## Split first

This is the single most important ordering decision in the whole notebook. Everything downstream — scaling, resampling, feature engineering that uses statistics of the data — has to be *fit* on `X_train` only and *applied* to `X_test`, never the reverse. Splitting first makes that easy to enforce.

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train fraud rate: {y_train.mean():.4%}  ({y_train.sum()} fraud / {len(y_train)})")
print(f"Test fraud rate:  {y_test.mean():.4%}  ({y_test.sum()} fraud / {len(y_test)})")

## Feature engineering

Same ideas as before (hour of day, log amount), but computed in a way that's safe to reuse: `Hour` is a deterministic function of `Time`, and `LogAmount` doesn't depend on any statistic fit from the data, so there's no leakage risk in adding them before or after the split.

In [ ]:
def add_features(frame):
    frame = frame.copy()
    frame['Hour'] = (frame['Time'] // 3600) % 24
    frame['LogAmount'] = np.log1p(frame['Amount'])
    return frame

X_train = add_features(X_train)
X_test = add_features(X_test)

## Preprocessing + model pipelines

`Amount`, `Time`, and `LogAmount` get scaled; the `V1`-`V28` PCA features are already roughly standardized and are passed through unchanged. Wrapping this in a `ColumnTransformer` inside a `Pipeline` means the scaler is *fit only on `X_train`* and the *same fitted transform* is reused on `X_test` and on any new data later — no more risk of a model seeing one scale at train time and a different scale at predict time.

Imbalance is handled with `class_weight='balanced'` (logistic regression, random forest) rather than undersampling the full dataset — that keeps every fraud *and* every legit example from the training set actually in the training set, and none of it leaks into the test set.

In [ ]:
numeric_to_scale = ['Time', 'Amount', 'LogAmount']
passthrough_cols = [c for c in X_train.columns if c not in numeric_to_scale]

preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), numeric_to_scale),
    ('passthrough', 'passthrough', passthrough_cols),
])

log_reg_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)),
])

rf_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    )),
])

## XGBoost

`scale_pos_weight` is XGBoost's equivalent of `class_weight='balanced'` — it's computed from the **training set only**.

In [ ]:
from xgboost import XGBClassifier

neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print(f"scale_pos_weight = {scale_pos_weight:.1f}")

xgb_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
    )),
])

## Cross-validate on the training set

Before touching the test set at all, compare models with stratified 5-fold CV on `X_train`/`y_train`. This gives an honest read on which model is worth taking to final evaluation, without spending the test set on model selection.

In [ ]:
models = {
    'Logistic Regression': log_reg_pipe,
    'Random Forest': rf_pipe,
    'XGBoost': xgb_pipe,
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ['precision', 'recall', 'f1', 'average_precision']

cv_results = {}
for name, pipe in models.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    cv_results[name] = {m: scores[f'test_{m}'].mean() for m in scoring}
    print(f"\n{name}")
    for m in scoring:
        print(f"  {m}: {scores[f'test_{m}'].mean():.3f} (+/- {scores[f'test_{m}'].std():.3f})")

pd.DataFrame(cv_results).T

## Final evaluation on the held-out test set

Fit each pipeline on the full training set and evaluate once on the untouched, realistically-imbalanced test set. This is the number that actually matters — it's what you'd see in production.

In [ ]:
fitted = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    y_pred = pipe.predict(X_test)
    print(f"\n=== {name} ===")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, digits=3))

## Threshold tuning

Default `predict()` uses a 0.5 cutoff, which isn't necessarily right for a problem where you might prefer to catch more fraud at the cost of more manual review, or vice versa. Use the precision-recall curve on the best model (from the eval above) to pick a threshold deliberately instead of accepting the default.

In [ ]:
best_name = 'XGBoost'  # update based on the CV / test results above
best_model = fitted[best_name]

y_scores = best_model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_scores)
ap = average_precision_score(y_test, y_scores)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, label=f'AP = {ap:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title(f'{best_name}: Precision-Recall Curve')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
    y_pred_t = (y_scores >= t).astype(int)
    print(f"\n--- Threshold: {t} ---")
    print(confusion_matrix(y_test, y_pred_t))
    print(classification_report(y_test, y_pred_t, digits=3))

## Feature importance (best tree-based model)

In [ ]:
clf = best_model.named_steps['clf']
feature_names = numeric_to_scale + passthrough_cols  # matches ColumnTransformer output order

if hasattr(clf, 'feature_importances_'):
    importances = pd.DataFrame({
        'Feature': feature_names,
        'Importance': clf.feature_importances_
    }).sort_values('Importance', ascending=False).head(15)

    plt.figure(figsize=(9, 6))
    sns.barplot(x='Importance', y='Feature', data=importances, hue='Feature', legend=False, palette='viridis')
    plt.title('Top 15 Feature Importances')
    plt.tight_layout()
    plt.show()

## Save the final pipeline

Save the *whole pipeline* (preprocessing + model together), not just the model. That way there's only ever one artifact to load, and it's structurally impossible to apply the wrong scaler at inference time — which is exactly the bug that caused most of the bad numbers in the original notebook.

In [ ]:
import joblib
joblib.dump(best_model, 'fraud_pipeline.pkl')

## Predicting a single transaction

In [ ]:
def predict_transaction(transaction_df, pipeline, threshold=0.5):
    """transaction_df: DataFrame with the same raw columns as the original data
    (Time, V1..V28, Amount) for one or more rows — no manual scaling needed,
    the pipeline handles it."""
    transaction_df = add_features(transaction_df)
    proba = pipeline.predict_proba(transaction_df)[:, 1]
    labels = np.where(proba >= threshold, 'Fraud', 'Not Fraud')
    return labels, proba

# Example:
# labels, proba = predict_transaction(X_test.iloc[[0]], best_model)
# print(labels, proba)

## Validating on truly unseen data

If you have a genuinely separate dataset (not another copy of the same file), load it here and run it through the exact same pipeline — no re-fitting scalers, no manual column selection.

In [ ]:
# new_df = pd.read_csv('creditcard2.csv')
# X_new = add_features(new_df.drop(columns=['Class']))
# y_new = new_df['Class']
# y_pred_new = best_model.predict(X_new)
# print(confusion_matrix(y_new, y_pred_new))
# print(classification_report(y_new, y_pred_new, digits=3))